# 🎵 StreamStay — AI-Based Customer Churn Prediction & Watch-Time Analytics
### Music Streaming Service · 12-Week Build Plan

---

**Project Summary**

| Task | Target |
|---|---|
| **Classify** | Subscriber churn (next 30 days) |
| **Predict** | Expected weekly watch-time (minutes) |
| **Segment** | Viewer taste segments (k-means) |
| **Deploy** | Churn-risk + engagement endpoint |

---

**12-Week Build Plan:**
- **W1** – Frame targets, EDA first look
- **W2** – Data ingestion, cleaning, leakage-safe pipeline
- **W3** – Linear regression baseline (watch-time), first MLflow run
- **W4** – Ridge/Lasso + logistic regression classifier
- **W5** – Cross-validation, one-standard-error rule, baseline milestone
- **W6** – Decision tree classifier, interpretable splits
- **W7** – Random forest, OOB error, permutation importance
- **W8** – XGBoost champion, SHAP explanations
- **W9** – k-means taste segmentation, PCA visualisation
- **W10** – Nested CV, calibration, imbalance-aware metrics, sealed test
- **W11** – Package champion, PSI drift guard
- **W12** – FastAPI deployment endpoint (containerised)

---
## ⚙️ 0 · Setup & Imports

In [ ]:
# ── Install required packages (run once) ──────────────────────────────────────
import subprocess, sys

PACKAGES = [
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn',
    'xgboost', 'shap', 'imbalanced-learn', 'mlflow', 'joblib',
    'fastapi', 'uvicorn', 'pydantic'
]

for pkg in PACKAGES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed.')

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import warnings, os, json, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, KFold,
    GridSearchCV, cross_validate
)
from sklearn.preprocessing import (
    StandardScaler, LabelEncoder, OrdinalEncoder, label_binarize
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, LogisticRegression,
    SGDRegressor
)
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, brier_score_loss, f1_score
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance

import xgboost as xgb
import shap
import joblib
import mlflow
import mlflow.sklearn

# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH  = 'streamstay_music_1200.csv'
MODEL_DIR  = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('✅ Imports OK')

---
## 📅 WEEK 1 — Frame Targets, First EDA Look

> **Goal:** Define the two targets — *subscriber churn (next 30 days)* (classification) and *expected weekly watch-time (minutes)* (regression). Estimate the irreducible-error floor and take a first exploratory look at anonymised viewing + subscription records.

In [ ]:
# ── W1.1 · Load & first look ──────────────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH)

print(f'Dataset shape : {df_raw.shape}')
print(f'Churn rate    : {df_raw["churn_next_30_days"].mean():.2%}  ({df_raw["churn_next_30_days"].sum()} churned)')
print(f'Watch-time (weekly) — mean: {df_raw["expected_weekly_listening_minutes"].mean():.1f} min, '
      f'std: {df_raw["expected_weekly_listening_minutes"].std():.1f} min')
display(df_raw.head())
display(df_raw.dtypes.rename('dtype').to_frame())

In [ ]:
# ── W1.2 · Target distributions ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Churn bar chart
churn_counts = df_raw['churn_next_30_days'].value_counts()
axes[0].bar(['Retained', 'Churned'], churn_counts.values,
            color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.5)
axes[0].set_title('TARGET 1 — Churn Distribution', fontweight='bold')
axes[0].set_ylabel('User count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Watch-time histogram
axes[1].hist(df_raw['expected_weekly_listening_minutes'], bins=40,
             color='#4C72B0', edgecolor='white', alpha=0.85)
axes[1].axvline(df_raw['expected_weekly_listening_minutes'].mean(),
                color='red', linestyle='--', label='Mean')
axes[1].axvline(df_raw['expected_weekly_listening_minutes'].median(),
                color='orange', linestyle='--', label='Median')
axes[1].set_title('TARGET 2 — Weekly Watch-Time Distribution', fontweight='bold')
axes[1].set_xlabel('Minutes / week')
axes[1].legend()

# Watch-time by churn group
df_raw.boxplot(column='expected_weekly_listening_minutes',
               by='churn_next_30_days', ax=axes[2],
               boxprops=dict(color='#4C72B0'),
               medianprops=dict(color='red', linewidth=2))
axes[2].set_title('Watch-Time by Churn Group', fontweight='bold')
axes[2].set_xlabel('Churn (0=No, 1=Yes)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('w1_target_distributions.png', bbox_inches='tight')
plt.show()
print('✅ W1 complete — targets framed.')

In [ ]:
# ── W1.3 · Irreducible-error floor estimate ───────────────────────────────────
# Naive baseline: always predict mean watch-time
naive_rmse = np.sqrt(mean_squared_error(
    df_raw['expected_weekly_listening_minutes'],
    np.full(len(df_raw), df_raw['expected_weekly_listening_minutes'].mean())
))
# Naive churn: always predict majority class
naive_acc  = 1 - df_raw['churn_next_30_days'].mean()

print('Irreducible-error floor estimates')
print('─' * 40)
print(f'  Regression naive RMSE  : {naive_rmse:.2f} min')
print(f'  Classifier naive acc   : {naive_acc:.2%} (majority class)')
print(f'  Classifier naive AUC   : 0.50 (random)')

---
## 📅 WEEK 2 — Data Ingestion, Cleaning & Leakage-Safe Pipeline

> **Goal:** Ingest anonymised viewing + subscription records; clean and validate viewing history, sessions, catalogue, plan, handle missingness and outliers, and lock a leakage-safe pipeline with documented EDA.

In [ ]:
# ── W2.1 · Deep EDA ──────────────────────────────────────────────────────────
df = df_raw.copy()

# Missing values
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('→ No missing values detected ✅')

# Outlier summary via IQR
num_cols = df.select_dtypes(include=np.number).columns.drop(
    ['churn_next_30_days', 'auto_renew', 'payment_failures', 'previous_cancellations']
)
outlier_info = []
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outlier_info.append({'feature': col, 'n_outliers': n_out,
                         'pct': f'{n_out/len(df):.1%}'})
display(pd.DataFrame(outlier_info).sort_values('n_outliers', ascending=False).head(10))

In [ ]:
# ── W2.2 · EDA — Categorical feature breakdowns ───────────────────────────────
cat_cols = ['subscription_type', 'platform', 'device_type', 'top_genre', 'country']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_by = df.groupby(col)['churn_next_30_days'].mean().sort_values(ascending=False)
    bars = axes[i].bar(churn_by.index, churn_by.values * 100,
                       color=sns.color_palette('muted', len(churn_by)),
                       edgecolor='white')
    axes[i].set_title(f'Churn Rate by {col}', fontweight='bold')
    axes[i].set_ylabel('Churn % ')
    axes[i].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    axes[i].tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, churn_by.values):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.2,
                     f'{val:.1%}', ha='center', fontsize=8)

# Hide last empty subplot
axes[-1].set_visible(False)
plt.suptitle('StreamStay — Churn Rate by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w2_categorical_eda.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── W2.3 · Correlation heatmap ────────────────────────────────────────────────
num_df = df.select_dtypes(include=np.number)
corr   = num_df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.4, cbar_kws={'shrink': 0.8})
plt.title('StreamStay — Feature Correlation Matrix', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('w2_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── W2.4 · Leakage-safe train/val/test split ──────────────────────────────────
# Features and targets
CAT_FEATURES = ['country', 'platform', 'device_type', 'subscription_type',
                'top_genre', 'top_artist']
NUM_FEATURES = ['age', 'daily_listening_minutes', 'songs_per_day',
                'sessions_per_week', 'avg_song_duration_sec', 'skip_rate_pct',
                'playlists_created', 'likes_per_week', 'unique_artists_per_week',
                'offline_usage_pct', 'recommendation_rating',
                'days_since_last_active', 'subscription_months',
                'monthly_fee', 'auto_renew', 'payment_failures',
                'previous_cancellations']

TARGET_CLF = 'churn_next_30_days'
TARGET_REG = 'expected_weekly_listening_minutes'

X = df.drop(columns=['user_id', TARGET_CLF, TARGET_REG])
y_clf = df[TARGET_CLF]
y_reg = df[TARGET_REG]

# 60 / 20 / 20 stratified split
X_tmp, X_test, y_clf_tmp, y_clf_test, y_reg_tmp, y_reg_test = train_test_split(
    X, y_clf, y_reg, test_size=0.20, random_state=RANDOM_STATE, stratify=y_clf
)
X_train, X_val, y_clf_train, y_clf_val, y_reg_train, y_reg_val = train_test_split(
    X_tmp, y_clf_tmp, y_reg_tmp, test_size=0.25,
    random_state=RANDOM_STATE, stratify=y_clf_tmp
)

print(f'Train : {X_train.shape[0]} rows  ({y_clf_train.mean():.2%} churn)')
print(f'Val   : {X_val.shape[0]} rows  ({y_clf_val.mean():.2%} churn)')
print(f'Test  : {X_test.shape[0]} rows  ({y_clf_test.mean():.2%} churn) ← SEALED')

In [ ]:
# ── W2.5 · Build shared pre-processing pipeline ───────────────────────────────
from sklearn.preprocessing import OneHotEncoder

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, NUM_FEATURES),
    ('cat', cat_pipe, CAT_FEATURES)
], remainder='drop')

# Fit on train only — no leakage
preprocessor.fit(X_train)
X_train_t = preprocessor.transform(X_train)
X_val_t   = preprocessor.transform(X_val)
X_test_t  = preprocessor.transform(X_test)

print(f'Preprocessed shape: {X_train_t.shape}  → {X_train_t.shape[1]} features')
print('✅ W2 complete — leakage-safe pipeline locked.')

---
## 📅 WEEK 3 — Linear Regression Baseline (Watch-Time) + First MLflow Run

> **Goal:** Build a linear-regression baseline for expected weekly watch-time with OLS, then gradient descent; log the first MLflow run.

In [ ]:
# ── W3.1 · OLS Linear Regression ─────────────────────────────────────────────
mlflow.set_experiment('StreamStay_WatchTime')

with mlflow.start_run(run_name='W3_OLS_LinearRegression'):
    ols = LinearRegression()
    ols.fit(X_train_t, y_reg_train)

    y_pred_val = ols.predict(X_val_t)
    rmse = np.sqrt(mean_squared_error(y_reg_val, y_pred_val))
    mae  = mean_absolute_error(y_reg_val, y_pred_val)
    r2   = r2_score(y_reg_val, y_pred_val)

    mlflow.log_metric('val_rmse', rmse)
    mlflow.log_metric('val_mae',  mae)
    mlflow.log_metric('val_r2',   r2)
    mlflow.sklearn.log_model(ols, 'ols_model')

    print('OLS Baseline — Validation Metrics')
    print(f'  RMSE : {rmse:.2f} min')
    print(f'  MAE  : {mae:.2f} min')
    print(f'  R²   : {r2:.4f}')
    print(f'  Naive RMSE was: {naive_rmse:.2f} min')
    print(f'  Improvement  : {(naive_rmse - rmse)/naive_rmse:.1%}')

In [ ]:
# ── W3.2 · Gradient Descent (SGD) ────────────────────────────────────────────
with mlflow.start_run(run_name='W3_SGD_GradientDescent'):
    sgd = SGDRegressor(max_iter=1000, tol=1e-4, learning_rate='adaptive',
                       eta0=0.01, random_state=RANDOM_STATE)
    sgd.fit(X_train_t, y_reg_train)

    y_pred_sgd = sgd.predict(X_val_t)
    rmse_sgd = np.sqrt(mean_squared_error(y_reg_val, y_pred_sgd))
    mae_sgd  = mean_absolute_error(y_reg_val, y_pred_sgd)
    r2_sgd   = r2_score(y_reg_val, y_pred_sgd)

    mlflow.log_metric('val_rmse', rmse_sgd)
    mlflow.log_metric('val_mae',  mae_sgd)
    mlflow.log_metric('val_r2',   r2_sgd)

    print('SGD Regression — Validation Metrics')
    print(f'  RMSE : {rmse_sgd:.2f} min')
    print(f'  MAE  : {mae_sgd:.2f} min')
    print(f'  R²   : {r2_sgd:.4f}')

In [ ]:
# ── W3.3 · Actual vs Predicted plot ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# OLS
axes[0].scatter(y_reg_val, ols.predict(X_val_t), alpha=0.5, color='#4C72B0', s=25)
mn, mx = y_reg_val.min(), y_reg_val.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual (min/week)'); axes[0].set_ylabel('Predicted (min/week)')
axes[0].set_title(f'OLS — Actual vs Predicted\nRMSE={rmse:.1f}, R²={r2:.3f}', fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_reg_val - ols.predict(X_val_t)
axes[1].hist(residuals, bins=35, color='#DD8452', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (min/week)'); axes[1].set_ylabel('Count')
axes[1].set_title('OLS — Residual Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('w3_regression_baseline.png', bbox_inches='tight')
plt.show()
print('✅ W3 complete — first MLflow run logged.')

---
## 📅 WEEK 4 — Ridge / Lasso + Logistic Regression Classifier

> **Goal:** Add ridge/lasso to the watch-time model; build a logistic-regression classifier for subscriber churn; engineer features from viewing history, sessions, catalogue, plan.

In [ ]:
# ── W4.1 · Feature Engineering ────────────────────────────────────────────────
def engineer_features(df_in):
    df_e = df_in.copy()
    # Engagement index
    df_e['engagement_score'] = (
        df_e['daily_listening_minutes'] * 0.4 +
        df_e['likes_per_week']           * 5  +
        df_e['playlists_created']        * 10 -
        df_e['skip_rate_pct']            * 2  -
        df_e['days_since_last_active']   * 3
    )
    # Loyalty index
    df_e['loyalty_index'] = (
        df_e['subscription_months'] * df_e['auto_renew'] -
        df_e['payment_failures']    * 10 -
        df_e['previous_cancellations'] * 15
    )
    # Session density
    df_e['session_density'] = (
        df_e['songs_per_day'] / (df_e['sessions_per_week'] / 7 + 1e-3)
    )
    # Catalogue breadth
    df_e['catalogue_breadth'] = df_e['unique_artists_per_week'] * df_e['playlists_created']
    # Paid flag
    df_e['is_paid'] = (df_e['monthly_fee'] > 0).astype(int)
    return df_e

NEW_FEATURES = ['engagement_score', 'loyalty_index', 'session_density',
                'catalogue_breadth', 'is_paid']

X_train_e = engineer_features(X_train)
X_val_e   = engineer_features(X_val)
X_test_e  = engineer_features(X_test)

print('New features added:', NEW_FEATURES)
X_train_e[NEW_FEATURES].describe().round(2)

In [ ]:
# ── W4.2 · Rebuild preprocessor with engineered features ─────────────────────
NUM_FEATURES_E = NUM_FEATURES + NEW_FEATURES

num_pipe_e = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_pipe_e = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor_e = ColumnTransformer([
    ('num', num_pipe_e, NUM_FEATURES_E),
    ('cat', cat_pipe_e, CAT_FEATURES)
], remainder='drop')

preprocessor_e.fit(X_train_e)
Xtr_e = preprocessor_e.transform(X_train_e)
Xva_e = preprocessor_e.transform(X_val_e)
Xte_e = preprocessor_e.transform(X_test_e)
print(f'Engineered feature matrix: {Xtr_e.shape}')

In [ ]:
# ── W4.3 · Ridge & Lasso regression ──────────────────────────────────────────
mlflow.set_experiment('StreamStay_WatchTime')
results_reg = {}

for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    for ModelClass, name in [(Ridge, 'Ridge'), (Lasso, 'Lasso')]:
        with mlflow.start_run(run_name=f'W4_{name}_a{alpha}'):
            m = ModelClass(alpha=alpha)
            m.fit(Xtr_e, y_reg_train)
            rmse_v = np.sqrt(mean_squared_error(y_reg_val, m.predict(Xva_e)))
            r2_v   = r2_score(y_reg_val, m.predict(Xva_e))
            mlflow.log_params({'model': name, 'alpha': alpha})
            mlflow.log_metrics({'val_rmse': rmse_v, 'val_r2': r2_v})
            results_reg[f'{name}_a{alpha}'] = {'rmse': rmse_v, 'r2': r2_v}

results_reg_df = pd.DataFrame(results_reg).T.sort_values('rmse')
print('Top 5 Ridge/Lasso configs:')
print(results_reg_df.head(5).to_string())

In [ ]:
# ── W4.4 · Best Ridge model ───────────────────────────────────────────────────
best_alpha_ridge = 1.0
ridge_best = Ridge(alpha=best_alpha_ridge)
ridge_best.fit(Xtr_e, y_reg_train)
rmse_ridge = np.sqrt(mean_squared_error(y_reg_val, ridge_best.predict(Xva_e)))
r2_ridge   = r2_score(y_reg_val, ridge_best.predict(Xva_e))
print(f'Best Ridge  α={best_alpha_ridge}  → RMSE={rmse_ridge:.2f}, R²={r2_ridge:.4f}')

In [ ]:
# ── W4.5 · Logistic Regression Churn Classifier ───────────────────────────────
mlflow.set_experiment('StreamStay_Churn')

with mlflow.start_run(run_name='W4_LogisticRegression'):
    lr_clf = LogisticRegression(
        C=1.0, class_weight='balanced',
        max_iter=1000, random_state=RANDOM_STATE
    )
    lr_clf.fit(Xtr_e, y_clf_train)

    y_prob_lr = lr_clf.predict_proba(Xva_e)[:, 1]
    y_pred_lr = lr_clf.predict(Xva_e)

    auc_lr = roc_auc_score(y_clf_val, y_prob_lr)
    ap_lr  = average_precision_score(y_clf_val, y_prob_lr)
    f1_lr  = f1_score(y_clf_val, y_pred_lr)

    mlflow.log_params({'model': 'LogisticRegression', 'C': 1.0,
                       'class_weight': 'balanced'})
    mlflow.log_metrics({'val_roc_auc': auc_lr, 'val_avg_precision': ap_lr,
                        'val_f1': f1_lr})
    mlflow.sklearn.log_model(lr_clf, 'lr_clf')

    print('Logistic Regression — Validation Metrics')
    print(f'  ROC-AUC          : {auc_lr:.4f}')
    print(f'  Avg Precision    : {ap_lr:.4f}')
    print(f'  F1 Score         : {f1_lr:.4f}')
    print()
    print(classification_report(y_clf_val, y_pred_lr,
                                 target_names=['Retained', 'Churned']))

In [ ]:
# ── W4.6 · ROC & PR curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
fpr, tpr, _ = roc_curve(y_clf_val, y_prob_lr)
axes[0].plot(fpr, tpr, lw=2, label=f'LR  AUC={auc_lr:.3f}')
axes[0].plot([0,1],[0,1],'k--',lw=1,label='Random')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve — Logistic Regression', fontweight='bold')
axes[0].legend()

# PR
prec, rec, _ = precision_recall_curve(y_clf_val, y_prob_lr)
axes[1].plot(rec, prec, lw=2, label=f'LR  AP={ap_lr:.3f}')
axes[1].axhline(y_clf_val.mean(), color='r', linestyle='--', label='Baseline')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('w4_lr_curves.png', bbox_inches='tight')
plt.show()
print('✅ W4 complete — Ridge/Lasso + Logistic Regression done.')

---
## 📅 WEEK 5 — Cross-Validation, One-Standard-Error Rule & Baseline Milestone

> **Goal:** Tune with cross-validation, apply the one-standard-error rule, and ship the honest linear/logistic baseline milestone.

In [ ]:
# ── W5.1 · CV for Ridge alpha selection (one-SE rule) ─────────────────────────
from sklearn.linear_model import RidgeCV

alphas = np.logspace(-3, 3, 50)
ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring='neg_root_mean_squared_error')
ridge_cv.fit(Xtr_e, y_reg_train)

print(f'RidgeCV best alpha : {ridge_cv.alpha_:.4f}')
rmse_ridgecv = np.sqrt(mean_squared_error(
    y_reg_val, ridge_cv.predict(Xva_e)))
print(f'Val RMSE           : {rmse_ridgecv:.2f} min')

In [ ]:
# ── W5.2 · K-Fold CV for Logistic Regression ──────────────────────────────────
from sklearn.pipeline import make_pipeline

# Combine preprocessing + model into a single pipeline for CV
X_combined = pd.concat([X_train_e, X_val_e], axis=0).reset_index(drop=True)
y_combined = pd.concat([y_clf_train, y_clf_val], axis=0).reset_index(drop=True)

cv_pipe = Pipeline([
    ('prep', preprocessor_e),
    ('clf',  LogisticRegression(C=1.0, class_weight='balanced',
                                max_iter=1000, random_state=RANDOM_STATE))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_validate(cv_pipe, X_combined, y_combined, cv=cv,
                            scoring=['roc_auc', 'average_precision'],
                            return_train_score=True)

print('5-Fold CV — Logistic Regression')
print(f'  ROC-AUC : {cv_scores["test_roc_auc"].mean():.4f} ± {cv_scores["test_roc_auc"].std():.4f}')
print(f'  Avg-Prec: {cv_scores["test_average_precision"].mean():.4f} ± {cv_scores["test_average_precision"].std():.4f}')

# One-standard-error rule: report best - 1SE
best_auc = cv_scores["test_roc_auc"].mean()
one_se_floor = best_auc - cv_scores["test_roc_auc"].std()
print(f'  1-SE floor AUC: {one_se_floor:.4f}')

In [ ]:
# ── W5.3 · Baseline Milestone Summary ─────────────────────────────────────────
print('='*60)
print('   BASELINE MILESTONE — WEEK 5 SUMMARY')
print('='*60)
print()
print('REGRESSION (Expected Weekly Watch-Time)')
print(f'  Naive RMSE (mean)  : {naive_rmse:.2f} min')
print(f'  OLS RMSE (val)     : {rmse:.2f} min')
print(f'  Ridge CV RMSE (val): {rmse_ridgecv:.2f} min')
print()
print('CLASSIFICATION (30-day Churn)')
print(f'  Naive AUC          : 0.500')
print(f'  LR val ROC-AUC     : {auc_lr:.4f}')
print(f'  LR 5-CV ROC-AUC    : {best_auc:.4f} ± {cv_scores["test_roc_auc"].std():.4f}')
print('='*60)
print('✅ W5 complete — Baseline milestone shipped.')

---
## 📅 WEEK 6 — Interpretable Decision Tree Classifier

> **Goal:** Grow an interpretable decision tree for subscriber churn and read its top splits on viewing history, sessions, catalogue, plan.

In [ ]:
# ── W6.1 · Decision Tree with depth tuning via CV ─────────────────────────────
from sklearn.tree import plot_tree

depth_scores = {}
for depth in range(2, 12):
    dt = Pipeline([
        ('prep', preprocessor_e),
        ('clf',  DecisionTreeClassifier(max_depth=depth,
                                        class_weight='balanced',
                                        random_state=RANDOM_STATE))
    ])
    score = cross_val_score(dt, X_combined, y_combined, cv=cv,
                             scoring='roc_auc').mean()
    depth_scores[depth] = score

best_depth = max(depth_scores, key=depth_scores.get)
print(f'Best tree depth: {best_depth}  (AUC={depth_scores[best_depth]:.4f})')

# Plot depth vs AUC
plt.figure(figsize=(8, 4))
plt.plot(list(depth_scores.keys()), list(depth_scores.values()),
         'o-', color='#4C72B0', linewidth=2, markersize=7)
plt.axvline(best_depth, color='red', linestyle='--', label=f'Best depth={best_depth}')
plt.xlabel('Max Tree Depth'); plt.ylabel('CV ROC-AUC')
plt.title('Decision Tree — Depth Tuning', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('w6_dt_depth_tuning.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── W6.2 · Train final Decision Tree ─────────────────────────────────────────
dt_final = DecisionTreeClassifier(max_depth=best_depth,
                                   class_weight='balanced',
                                   min_samples_leaf=10,
                                   random_state=RANDOM_STATE)
dt_final.fit(Xtr_e, y_clf_train)

y_prob_dt = dt_final.predict_proba(Xva_e)[:, 1]
auc_dt = roc_auc_score(y_clf_val, y_prob_dt)
ap_dt  = average_precision_score(y_clf_val, y_prob_dt)
print(f'Decision Tree  → Val ROC-AUC: {auc_dt:.4f},  Avg Precision: {ap_dt:.4f}')

In [ ]:
# ── W6.3 · Visualise the tree (depth 3 for readability) ──────────────────────
# Get feature names after one-hot encoding
ohe_features = (preprocessor_e
                .named_transformers_['cat']
                .named_steps['ohe']
                .get_feature_names_out(CAT_FEATURES)
                .tolist())
feature_names_all = NUM_FEATURES_E + ohe_features

dt_viz = DecisionTreeClassifier(max_depth=3, class_weight='balanced',
                                 min_samples_leaf=10, random_state=RANDOM_STATE)
dt_viz.fit(Xtr_e, y_clf_train)

plt.figure(figsize=(20, 8))
plot_tree(dt_viz, feature_names=feature_names_all,
          class_names=['Retained', 'Churned'],
          filled=True, rounded=True, fontsize=9,
          impurity=False, proportion=True)
plt.title('Decision Tree — Churn Prediction (depth=3, interpretable view)',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('w6_decision_tree.png', bbox_inches='tight', dpi=130)
plt.show()

# Top splits text
print('\nTop decision splits:')
print(export_text(dt_viz, feature_names=feature_names_all, max_depth=3))
print('✅ W6 complete — Decision Tree analysed.')

In [ ]:
# ── W6.4 · Feature importances from DT ───────────────────────────────────────
fi_dt = pd.Series(dt_final.feature_importances_, index=feature_names_all)
fi_dt = fi_dt.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
fi_dt.sort_values().plot(kind='barh', color='#4C72B0', edgecolor='white')
plt.xlabel('Gini Importance')
plt.title('Decision Tree — Top 15 Feature Importances', fontweight='bold')
plt.tight_layout()
plt.savefig('w6_dt_feature_importance.png', bbox_inches='tight')
plt.show()

---
## 📅 WEEK 7 — Random Forest: OOB Error & Permutation Importance

> **Goal:** Train a random forest — cut variance, read OOB error, and rank features by permutation importance.

In [ ]:
# ── W7.1 · Random Forest Classifier ──────────────────────────────────────────
mlflow.set_experiment('StreamStay_Churn')

with mlflow.start_run(run_name='W7_RandomForest'):
    rf_clf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        max_features='sqrt',
        class_weight='balanced',
        oob_score=True,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    rf_clf.fit(Xtr_e, y_clf_train)

    y_prob_rf = rf_clf.predict_proba(Xva_e)[:, 1]
    auc_rf = roc_auc_score(y_clf_val, y_prob_rf)
    ap_rf  = average_precision_score(y_clf_val, y_prob_rf)
    f1_rf  = f1_score(y_clf_val, rf_clf.predict(Xva_e))

    mlflow.log_params({'n_estimators': 300, 'max_features': 'sqrt',
                       'class_weight': 'balanced'})
    mlflow.log_metrics({'oob_score': rf_clf.oob_score_,
                        'val_roc_auc': auc_rf,
                        'val_avg_precision': ap_rf,
                        'val_f1': f1_rf})

    print('Random Forest — Results')
    print(f'  OOB Score (accuracy) : {rf_clf.oob_score_:.4f}')
    print(f'  Val ROC-AUC          : {auc_rf:.4f}')
    print(f'  Val Avg Precision    : {ap_rf:.4f}')
    print(f'  Val F1               : {f1_rf:.4f}')

In [ ]:
# ── W7.2 · OOB error vs n_estimators curve ────────────────────────────────────
oob_errors = []
for n in range(10, 310, 20):
    rf_tmp = RandomForestClassifier(
        n_estimators=n, max_features='sqrt',
        class_weight='balanced', oob_score=True,
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_tmp.fit(Xtr_e, y_clf_train)
    oob_errors.append(1 - rf_tmp.oob_score_)

plt.figure(figsize=(9, 4))
plt.plot(range(10, 310, 20), oob_errors, 'o-', color='#4C72B0', linewidth=2)
plt.xlabel('Number of Trees'); plt.ylabel('OOB Error Rate')
plt.title('Random Forest — OOB Error vs N Estimators', fontweight='bold')
plt.tight_layout()
plt.savefig('w7_oob_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── W7.3 · Permutation Importance ────────────────────────────────────────────
perm_imp = permutation_importance(
    rf_clf, Xva_e, y_clf_val, n_repeats=30,
    scoring='roc_auc', random_state=RANDOM_STATE, n_jobs=-1
)
perm_df = pd.DataFrame({
    'feature': feature_names_all,
    'importance_mean': perm_imp.importances_mean,
    'importance_std':  perm_imp.importances_std
}).sort_values('importance_mean', ascending=False).head(20)

plt.figure(figsize=(10, 7))
plt.barh(perm_df['feature'][::-1], perm_df['importance_mean'][::-1],
         xerr=perm_df['importance_std'][::-1],
         color='#4C72B0', edgecolor='white', capsize=4)
plt.xlabel('Mean AUC decrease')
plt.title('Random Forest — Top 20 Permutation Importances (Churn)', fontweight='bold')
plt.tight_layout()
plt.savefig('w7_permutation_importance.png', bbox_inches='tight')
plt.show()
print('✅ W7 complete — Random Forest built and analysed.')

In [ ]:
# ── W7.4 · Random Forest Regressor (Watch-Time) ───────────────────────────────
mlflow.set_experiment('StreamStay_WatchTime')

with mlflow.start_run(run_name='W7_RandomForestRegressor'):
    rf_reg = RandomForestRegressor(
        n_estimators=300, max_features='sqrt',
        min_samples_leaf=5, oob_score=True,
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_reg.fit(Xtr_e, y_reg_train)

    rmse_rf_reg = np.sqrt(mean_squared_error(
        y_reg_val, rf_reg.predict(Xva_e)))
    r2_rf_reg   = r2_score(y_reg_val, rf_reg.predict(Xva_e))

    mlflow.log_metrics({'val_rmse': rmse_rf_reg, 'val_r2': r2_rf_reg,
                        'oob_r2': rf_reg.oob_score_})
    print(f'RF Regressor → Val RMSE={rmse_rf_reg:.2f}, R²={r2_rf_reg:.4f}, OOB R²={rf_reg.oob_score_:.4f}')

---
## 📅 WEEK 8 — XGBoost Champion + SHAP Explanations

> **Goal:** Build the XGBoost champion; tune with early stopping and explain predictions with SHAP.

In [ ]:
# ── W8.1 · XGBoost Classifier (Churn) ────────────────────────────────────────
mlflow.set_experiment('StreamStay_Churn')

scale_pos = int(np.sum(y_clf_train == 0) / np.sum(y_clf_train == 1))
print(f'scale_pos_weight: {scale_pos}')

xgb_clf = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    scale_pos_weight=scale_pos,
    eval_metric='auc',
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    use_label_encoder=False,
    verbosity=0
)

xgb_clf.fit(
    Xtr_e, y_clf_train,
    eval_set=[(Xva_e, y_clf_val)],
    verbose=False
)

y_prob_xgb = xgb_clf.predict_proba(Xva_e)[:, 1]
auc_xgb = roc_auc_score(y_clf_val, y_prob_xgb)
ap_xgb  = average_precision_score(y_clf_val, y_prob_xgb)
f1_xgb  = f1_score(y_clf_val, xgb_clf.predict(Xva_e))

print(f'XGBoost best round: {xgb_clf.best_iteration}')
print(f'Val ROC-AUC       : {auc_xgb:.4f}')
print(f'Val Avg Precision : {ap_xgb:.4f}')
print(f'Val F1            : {f1_xgb:.4f}')

In [ ]:
# ── W8.2 · XGBoost Regressor (Watch-Time) ────────────────────────────────────
mlflow.set_experiment('StreamStay_WatchTime')

xgb_reg = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50,
    eval_metric='rmse',
    random_state=RANDOM_STATE,
    verbosity=0
)
xgb_reg.fit(
    Xtr_e, y_reg_train,
    eval_set=[(Xva_e, y_reg_val)],
    verbose=False
)

rmse_xgb_reg = np.sqrt(mean_squared_error(y_reg_val, xgb_reg.predict(Xva_e)))
r2_xgb_reg   = r2_score(y_reg_val, xgb_reg.predict(Xva_e))
print(f'XGBoost Reg → best_round={xgb_reg.best_iteration}, '
      f'Val RMSE={rmse_xgb_reg:.2f}, R²={r2_xgb_reg:.4f}')

In [ ]:
# ── W8.3 · SHAP Analysis ─────────────────────────────────────────────────────
explainer = shap.TreeExplainer(xgb_clf)
shap_values = explainer(Xva_e)

# SHAP feature names
shap_feature_names = feature_names_all

# Summary beeswarm
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values.values, Xva_e,
                  feature_names=shap_feature_names,
                  show=False, max_display=20)
plt.title('SHAP — Feature Impact on Churn Probability', fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('w8_shap_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── W8.4 · SHAP bar plot (mean absolute) ─────────────────────────────────────
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values.values, Xva_e,
                  feature_names=shap_feature_names,
                  plot_type='bar', show=False, max_display=15)
plt.title('SHAP — Mean |SHAP| Value (Global Importance)', fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('w8_shap_bar.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── W8.5 · SHAP waterfall (single high-risk subscriber) ─────────────────────
# Pick highest churn probability subscriber from validation set
high_risk_idx = np.argmax(y_prob_xgb)
plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_values[high_risk_idx], max_display=12, show=False)
plt.title(f'SHAP Waterfall — High-Risk Subscriber (p_churn={y_prob_xgb[high_risk_idx]:.3f})',
          fontweight='bold')
plt.tight_layout()
plt.savefig('w8_shap_waterfall.png', bbox_inches='tight')
plt.show()
print('✅ W8 complete — XGBoost champion + SHAP explanations done.')

In [ ]:
# ── W8.6 · Model comparison scorecard ────────────────────────────────────────
models_comparison = pd.DataFrame([
    {'Model': 'Logistic Regression',  'ROC-AUC': auc_lr,  'Avg Precision': ap_lr,  'F1': f1_lr},
    {'Model': 'Decision Tree',        'ROC-AUC': auc_dt,  'Avg Precision': ap_dt,  'F1': f1_score(y_clf_val, dt_final.predict(Xva_e))},
    {'Model': 'Random Forest',        'ROC-AUC': auc_rf,  'Avg Precision': ap_rf,  'F1': f1_rf},
    {'Model': 'XGBoost (Champion)',   'ROC-AUC': auc_xgb, 'Avg Precision': ap_xgb, 'F1': f1_xgb},
]).sort_values('ROC-AUC', ascending=False)

print('\n🏆 CHURN CLASSIFIER COMPARISON')
print(models_comparison.to_string(index=False, float_format='{:.4f}'.format))

reg_comparison = pd.DataFrame([
    {'Model': 'OLS',          'RMSE': rmse,         'R²': r2},
    {'Model': 'Ridge CV',     'RMSE': rmse_ridgecv, 'R²': r2_score(y_reg_val, ridge_cv.predict(Xva_e))},
    {'Model': 'RF Regressor', 'RMSE': rmse_rf_reg,  'R²': r2_rf_reg},
    {'Model': 'XGBoost Reg',  'RMSE': rmse_xgb_reg, 'R²': r2_xgb_reg},
]).sort_values('RMSE')

print('\n🏆 WATCH-TIME REGRESSOR COMPARISON')
print(reg_comparison.to_string(index=False, float_format='{:.4f}'.format))

---
## 📅 WEEK 9 — K-Means Taste Segmentation & PCA Visualisation

> **Goal:** Cluster viewers into taste segments (k-means on genre/behaviour, PCA for structure) and use the segment as a churn and engagement feature.

In [ ]:
# ── W9.1 · Prepare clustering features ───────────────────────────────────────
# Use all training features (preprocessed + engineered)
X_all_e = engineer_features(X)
X_all_t  = preprocessor_e.transform(X_all_e)

# Find optimal k via inertia elbow + silhouette
from sklearn.metrics import silhouette_score

inertias     = []
silhouettes  = []
K_range      = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_all_t)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_all_t, labels,
                                        sample_size=500, random_state=RANDOM_STATE))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(K_range, inertias, 'o-', color='#4C72B0', linewidth=2)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia (SSE)')
axes[0].set_title('Elbow Method', fontweight='bold')

axes[1].plot(K_range, silhouettes, 's-', color='#DD8452', linewidth=2)
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis', fontweight='bold')

plt.suptitle('K-Means — Optimal K Selection', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('w9_kmeans_tuning.png', bbox_inches='tight')
plt.show()

BEST_K = K_range[np.argmax(silhouettes)]
print(f'Best K by silhouette: {BEST_K}')

In [ ]:
# ── W9.2 · Fit final K-Means ──────────────────────────────────────────────────
BEST_K = max(BEST_K, 4)  # ensure at least 4 taste segments
kmeans = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=20)
cluster_labels = kmeans.fit_predict(X_all_t)
df['taste_segment'] = cluster_labels

print(f'Cluster distribution:\n{pd.Series(cluster_labels).value_counts().sort_index()}')

In [ ]:
# ── W9.3 · PCA 2D visualisation ──────────────────────────────────────────────
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_all_t)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['segment']      = cluster_labels
pca_df['churn']        = y_clf.values
pca_df['watch_time']   = y_reg.values

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Coloured by segment
palette = sns.color_palette('tab10', BEST_K)
for seg in range(BEST_K):
    mask = pca_df['segment'] == seg
    axes[0].scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
                    c=[palette[seg]], label=f'Segment {seg}', alpha=0.6, s=20)
axes[0].set_title(f'PCA — Taste Segments (K={BEST_K})', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
axes[0].legend(markerscale=2, fontsize=9)

# Coloured by churn
colours = pca_df['churn'].map({0: '#4C72B0', 1: '#DD8452'})
axes[1].scatter(pca_df['PC1'], pca_df['PC2'], c=colours, alpha=0.5, s=20)
from matplotlib.patches import Patch
axes[1].legend(handles=[
    Patch(color='#4C72B0', label='Retained'),
    Patch(color='#DD8452', label='Churned')
])
axes[1].set_title('PCA — Churn overlay', fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')

plt.tight_layout()
plt.savefig('w9_pca_segments.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── W9.4 · Segment profile & churn by segment ────────────────────────────────
seg_profile = df.groupby('taste_segment').agg(
    n_users         = ('user_id', 'count'),
    churn_rate      = ('churn_next_30_days', 'mean'),
    avg_watchtime   = ('expected_weekly_listening_minutes', 'mean'),
    avg_daily_min   = ('daily_listening_minutes', 'mean'),
    avg_skip_rate   = ('skip_rate_pct', 'mean'),
    top_genre       = ('top_genre', lambda x: x.value_counts().index[0]),
    top_sub         = ('subscription_type', lambda x: x.value_counts().index[0])
).round(3)

print('\n🎯 TASTE SEGMENT PROFILES')
display(seg_profile)

# Assign descriptive labels
seg_names = {i: f'Segment {i}' for i in range(BEST_K)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
seg_profile['churn_rate'].plot(kind='bar', ax=axes[0],
    color=sns.color_palette('tab10', BEST_K), edgecolor='white')
axes[0].set_title('Churn Rate by Taste Segment', fontweight='bold')
axes[0].set_ylabel('Churn Rate'); axes[0].set_xlabel('Segment')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))

seg_profile['avg_watchtime'].plot(kind='bar', ax=axes[1],
    color=sns.color_palette('tab10', BEST_K), edgecolor='white')
axes[1].set_title('Avg Watch-Time by Taste Segment', fontweight='bold')
axes[1].set_ylabel('Minutes / week'); axes[1].set_xlabel('Segment')

plt.tight_layout()
plt.savefig('w9_segment_profiles.png', bbox_inches='tight')
plt.show()
print('✅ W9 complete — Taste segments identified and profiled.')

In [ ]:
# ── W9.5 · Add segment as feature and retrain XGBoost ─────────────────────────
# Add segment label to all splits
X_train_seg = X_train_e.copy()
X_val_seg   = X_val_e.copy()
X_test_seg  = X_test_e.copy()

X_train_seg['taste_segment'] = kmeans.predict(
    preprocessor_e.transform(X_train_e))
X_val_seg['taste_segment']   = kmeans.predict(
    preprocessor_e.transform(X_val_e))
X_test_seg['taste_segment']  = kmeans.predict(
    preprocessor_e.transform(X_test_e))

NUM_FEATURES_SEG = NUM_FEATURES_E + ['taste_segment']

preprocessor_seg = ColumnTransformer([
    ('num', num_pipe_e, NUM_FEATURES_SEG),
    ('cat', cat_pipe_e, CAT_FEATURES)
], remainder='drop')

preprocessor_seg.fit(X_train_seg)
Xtr_seg = preprocessor_seg.transform(X_train_seg)
Xva_seg = preprocessor_seg.transform(X_val_seg)
Xte_seg = preprocessor_seg.transform(X_test_seg)

xgb_seg = xgb.XGBClassifier(
    n_estimators=1000, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    scale_pos_weight=scale_pos, eval_metric='auc',
    early_stopping_rounds=50, random_state=RANDOM_STATE,
    use_label_encoder=False, verbosity=0
)
xgb_seg.fit(Xtr_seg, y_clf_train,
            eval_set=[(Xva_seg, y_clf_val)], verbose=False)

auc_seg = roc_auc_score(y_clf_val, xgb_seg.predict_proba(Xva_seg)[:, 1])
print(f'XGBoost + Segment feature → Val ROC-AUC: {auc_seg:.4f}')
print(f'Improvement over no-segment: {auc_seg - auc_xgb:+.4f}')

---
## 📅 WEEK 10 — Rigorous Evaluation: Nested CV, Calibration & Sealed Test

> **Goal:** Evaluate rigorously — nested CV, probability calibration, imbalance-aware metrics, and a significance test on the sealed test set.

In [ ]:
# ── W10.1 · Nested Cross-Validation ──────────────────────────────────────────
from sklearn.model_selection import cross_val_score

print('Running nested CV — this may take a minute...')

X_dev_e = pd.concat([X_train_e, X_val_e]).reset_index(drop=True)
y_dev   = pd.concat([y_clf_train, y_clf_val]).reset_index(drop=True)

# Inner CV for HP tuning, outer for unbiased estimate
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_ncv_pipe = Pipeline([
    ('prep', preprocessor_e),
    ('clf',  xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, scale_pos_weight=scale_pos,
        eval_metric='auc', use_label_encoder=False,
        verbosity=0, random_state=RANDOM_STATE
    ))
])

nested_scores = cross_val_score(
    xgb_ncv_pipe, X_dev_e, y_dev,
    cv=outer_cv, scoring='roc_auc', n_jobs=-1
)
print(f'Nested CV ROC-AUC: {nested_scores.mean():.4f} ± {nested_scores.std():.4f}')
print(f'Individual folds : {nested_scores.round(4)}')

In [ ]:
# ── W10.2 · Probability Calibration ──────────────────────────────────────────
# Calibrate XGBoost with Platt scaling (sigmoid) and isotonic regression
cal_sigmoid  = CalibratedClassifierCV(
    xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, scale_pos_weight=scale_pos,
        eval_metric='auc', use_label_encoder=False,
        verbosity=0, random_state=RANDOM_STATE
    ),
    method='sigmoid', cv=3
)
cal_isotonic = CalibratedClassifierCV(
    xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, scale_pos_weight=scale_pos,
        eval_metric='auc', use_label_encoder=False,
        verbosity=0, random_state=RANDOM_STATE
    ),
    method='isotonic', cv=3
)

cal_sigmoid.fit(Xtr_e,  y_clf_train)
cal_isotonic.fit(Xtr_e, y_clf_train)

# Calibration curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, proba in [
    ('Uncalibrated XGB',    y_prob_xgb),
    ('Platt (sigmoid)',     cal_sigmoid.predict_proba(Xva_e)[:, 1]),
    ('Isotonic',            cal_isotonic.predict_proba(Xva_e)[:, 1])
]:
    frac_pos, mean_pred = calibration_curve(y_clf_val, proba, n_bins=8)
    ax.plot(mean_pred, frac_pos, 's-', label=name)

ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curves — XGBoost Churn', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('w10_calibration.png', bbox_inches='tight')
plt.show()

# Brier scores
for name, proba in [
    ('Uncalibrated XGB', y_prob_xgb),
    ('Platt',            cal_sigmoid.predict_proba(Xva_e)[:, 1]),
    ('Isotonic',         cal_isotonic.predict_proba(Xva_e)[:, 1])
]:
    bs = brier_score_loss(y_clf_val, proba)
    print(f'{name:25s} Brier={bs:.4f}')

In [ ]:
# ── W10.3 · Imbalance-aware metrics ──────────────────────────────────────────
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
Xtr_smote, y_clf_train_smote = smote.fit_resample(Xtr_e, y_clf_train)
print(f'After SMOTE: {pd.Series(y_clf_train_smote).value_counts().to_dict()}')

xgb_smote = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, eval_metric='auc',
    use_label_encoder=False, verbosity=0, random_state=RANDOM_STATE
)
xgb_smote.fit(Xtr_smote, y_clf_train_smote)
auc_smote = roc_auc_score(y_clf_val, xgb_smote.predict_proba(Xva_e)[:, 1])
print(f'SMOTE + XGBoost → Val ROC-AUC: {auc_smote:.4f}')

In [ ]:
# ── W10.4 · SEALED TEST SET EVALUATION ───────────────────────────────────────
print('='*60)
print('   🔒 SEALED TEST SET — FINAL EVALUATION')
print('='*60)

# Champion: XGBoost (with segment feature)
y_prob_test  = xgb_seg.predict_proba(Xte_seg)[:, 1]
y_pred_test  = xgb_seg.predict(Xte_seg)

test_auc = roc_auc_score(y_clf_test, y_prob_test)
test_ap  = average_precision_score(y_clf_test, y_prob_test)
test_f1  = f1_score(y_clf_test, y_pred_test)
test_bs  = brier_score_loss(y_clf_test, y_prob_test)

print(f'\nCHURN CLASSIFIER (XGBoost + Segments)')
print(f'  Test ROC-AUC       : {test_auc:.4f}')
print(f'  Test Avg Precision : {test_ap:.4f}')
print(f'  Test F1            : {test_f1:.4f}')
print(f'  Test Brier Score   : {test_bs:.4f}')
print()
print(classification_report(y_clf_test, y_pred_test,
                             target_names=['Retained', 'Churned']))

# Watch-time regressor
y_reg_pred_test = xgb_reg.predict(Xte_e)
test_rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred_test))
test_r2   = r2_score(y_reg_test, y_reg_pred_test)

print(f'WATCH-TIME REGRESSOR (XGBoost)')
print(f'  Test RMSE          : {test_rmse:.2f} min')
print(f'  Test R²            : {test_r2:.4f}')
print('='*60)

In [ ]:
# ── W10.5 · Confusion matrix & final ROC on test ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y_clf_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'])
axes[0].set_title(f'Confusion Matrix — Test Set\n(AUC={test_auc:.3f})', fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

# ROC
fpr_t, tpr_t, _ = roc_curve(y_clf_test, y_prob_test)
axes[1].plot(fpr_t, tpr_t, lw=2.5, label=f'XGBoost AUC={test_auc:.3f}')
axes[1].plot([0,1],[0,1],'k--',lw=1, label='Random')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Sealed Test Set', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('w10_test_evaluation.png', bbox_inches='tight')
plt.show()
print('✅ W10 complete — Rigorous evaluation done.')

---
## 📅 WEEK 11 — Package Champion + PSI Drift Guard

> **Goal:** Package the champion into one versioned artefact with a shared feature module and a PSI drift guard.

In [ ]:
# ── W11.1 · Save champion artefacts ──────────────────────────────────────────
import joblib
from datetime import datetime

VERSION = datetime.now().strftime('%Y%m%d')

ARTEFACT = {
    'version'        : VERSION,
    'preprocessor_e' : preprocessor_e,
    'preprocessor_seg': preprocessor_seg,
    'kmeans'         : kmeans,
    'xgb_clf'        : xgb_seg,        # churn champion
    'xgb_reg'        : xgb_reg,        # watch-time champion
    'feature_names'  : feature_names_all,
    'num_features'   : NUM_FEATURES_E,
    'cat_features'   : CAT_FEATURES,
    'seg_features'   : NUM_FEATURES_SEG,
    'BEST_K'         : BEST_K,
    'metrics'        : {
        'churn_test_auc'   : round(test_auc, 4),
        'churn_test_f1'    : round(test_f1, 4),
        'churn_test_ap'    : round(test_ap, 4),
        'watchtime_test_rmse': round(test_rmse, 2),
        'watchtime_test_r2'  : round(test_r2, 4)
    }
}

artefact_path = os.path.join(MODEL_DIR, f'streamstay_champion_v{VERSION}.pkl')
joblib.dump(ARTEFACT, artefact_path)
print(f'✅ Champion artefact saved → {artefact_path}')
print(f'   Size: {os.path.getsize(artefact_path)/1e6:.2f} MB')

In [ ]:
# ── W11.2 · PSI (Population Stability Index) Drift Guard ─────────────────────
def compute_psi(expected, actual, n_bins=10, eps=1e-6):
    """
    Compute Population Stability Index between expected (train)
    and actual (new) distributions.
    PSI < 0.10 → Stable
    PSI 0.10–0.25 → Slight drift
    PSI > 0.25  → Significant drift — retrain!
    """
    breakpoints = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    breakpoints = np.unique(breakpoints)
    
    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts   = np.histogram(actual,   bins=breakpoints)[0]
    
    expected_pct = expected_counts / (len(expected) + eps)
    actual_pct   = actual_counts   / (len(actual)   + eps)
    
    expected_pct = np.clip(expected_pct, eps, None)
    actual_pct   = np.clip(actual_pct,   eps, None)
    
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

def psi_report(X_train_mat, X_new_mat, feature_names, threshold=0.10):
    """Report PSI for each numeric feature."""
    results = []
    n_num = len(NUM_FEATURES_E)  # only numeric features
    for i, feat in enumerate(feature_names[:n_num]):
        psi = compute_psi(X_train_mat[:, i], X_new_mat[:, i])
        flag = '🚨 DRIFT' if psi > 0.25 else ('⚠️  Monitor' if psi > threshold else '✅ Stable')
        results.append({'feature': feat, 'PSI': round(psi, 4), 'status': flag})
    return pd.DataFrame(results).sort_values('PSI', ascending=False)

# Simulate production data = test set
psi_df = psi_report(Xtr_e, Xte_e, feature_names_all)
print('PSI Drift Guard — Train vs Test (simulated production)')
display(psi_df.head(15))

max_psi = psi_df['PSI'].max()
if max_psi > 0.25:
    print(f'\n🚨 DRIFT ALERT: Max PSI={max_psi:.4f} — schedule retraining!')
else:
    print(f'\n✅ No significant drift detected (Max PSI={max_psi:.4f})')

print('\n✅ W11 complete — Champion packaged + PSI drift guard active.')

In [ ]:
# ── W11.3 · PSI Visualisation ────────────────────────────────────────────────
top_psi = psi_df.head(12).sort_values('PSI')
colours = ['#DD8452' if p > 0.25 else '#FFBE7B' if p > 0.10 else '#4C72B0'
           for p in top_psi['PSI']]

plt.figure(figsize=(10, 5))
plt.barh(top_psi['feature'], top_psi['PSI'], color=colours, edgecolor='white')
plt.axvline(0.10, color='orange', linestyle='--', lw=1.5, label='Monitor (0.10)')
plt.axvline(0.25, color='red',    linestyle='--', lw=1.5, label='Drift (0.25)')
plt.xlabel('PSI'); plt.title('PSI Drift Guard — Feature Stability', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('w11_psi_drift.png', bbox_inches='tight')
plt.show()

---
## 📅 WEEK 12 — FastAPI Deployment Endpoint

> **Goal:** Deploy a churn-risk + engagement endpoint as a containerised FastAPI service with `/health`, `/metrics`, drift monitoring and a retraining trigger.

In [ ]:
# ── W12.1 · Write FastAPI app ─────────────────────────────────────────────────
fastapi_code = '''"""
StreamStay ML API — Week 12 Capstone
Endpoints:
  GET  /health            — liveness probe
  GET  /metrics           — model performance + PSI drift
  POST /predict           — churn probability + watch-time + segment
  POST /retrain_trigger   — schedule retraining if drift detected
"""
import os
import numpy as np
import pandas as pd
import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime

# ── Load champion artefact ────────────────────────────────────────────────────
MODEL_DIR    = "models"
artefact_files = sorted([f for f in os.listdir(MODEL_DIR) if f.endswith(".pkl")])
ARTEFACT_PATH  = os.path.join(MODEL_DIR, artefact_files[-1])
BUNDLE = joblib.load(ARTEFACT_PATH)

preprocessor_e   = BUNDLE["preprocessor_e"]
preprocessor_seg = BUNDLE["preprocessor_seg"]
kmeans           = BUNDLE["kmeans"]
xgb_clf          = BUNDLE["xgb_clf"]
xgb_reg          = BUNDLE["xgb_reg"]
METRICS          = BUNDLE["metrics"]
VERSION          = BUNDLE["version"]
NUM_FEATURES     = BUNDLE["num_features"]
CAT_FEATURES     = BUNDLE["cat_features"]
SEG_FEATURES     = BUNDLE["seg_features"]

app = FastAPI(
    title="StreamStay ML API",
    description="Churn risk + watch-time + taste segment prediction",
    version="1.0.0"
)

# ── Request / Response schemas ────────────────────────────────────────────────
class SubscriberFeatures(BaseModel):
    age: int = Field(..., ge=13, le=120)
    country: str
    platform: str
    device_type: str
    subscription_type: str
    top_genre: str
    top_artist: str
    daily_listening_minutes: float
    songs_per_day: float
    sessions_per_week: float
    avg_song_duration_sec: float
    skip_rate_pct: float
    playlists_created: int
    likes_per_week: int
    unique_artists_per_week: int
    offline_usage_pct: float
    recommendation_rating: float
    days_since_last_active: int
    subscription_months: int
    monthly_fee: float
    auto_renew: int
    payment_failures: int
    previous_cancellations: int

class PredictionResponse(BaseModel):
    churn_probability: float
    churn_risk_label: str
    expected_weekly_watch_time_min: float
    taste_segment: int
    model_version: str
    timestamp: str

# ── Helper: build feature row ─────────────────────────────────────────────────
def build_feature_row(sub: SubscriberFeatures) -> pd.DataFrame:
    row = pd.DataFrame([sub.dict()])
    row["engagement_score"] = (
        row["daily_listening_minutes"] * 0.4 +
        row["likes_per_week"]           * 5  +
        row["playlists_created"]        * 10 -
        row["skip_rate_pct"]            * 2  -
        row["days_since_last_active"]   * 3
    )
    row["loyalty_index"] = (
        row["subscription_months"] * row["auto_renew"] -
        row["payment_failures"]    * 10 -
        row["previous_cancellations"] * 15
    )
    row["session_density"]  = row["songs_per_day"] / (row["sessions_per_week"] / 7 + 1e-3)
    row["catalogue_breadth"] = row["unique_artists_per_week"] * row["playlists_created"]
    row["is_paid"]           = (row["monthly_fee"] > 0).astype(int)
    return row

# ── Endpoints ─────────────────────────────────────────────────────────────────
@app.get("/health", tags=["monitoring"])
def health_check():
    return {"status": "healthy", "model_version": VERSION,
            "timestamp": datetime.utcnow().isoformat()}

@app.get("/metrics", tags=["monitoring"])
def model_metrics():
    return {
        "model_version" : VERSION,
        "test_metrics"  : METRICS,
        "artefact_path" : ARTEFACT_PATH,
        "loaded_at"     : datetime.utcnow().isoformat()
    }

@app.post("/predict", response_model=PredictionResponse, tags=["prediction"])
def predict(sub: SubscriberFeatures):
    try:
        row_e = build_feature_row(sub)

        # Base transform
        X_base = preprocessor_e.transform(row_e)

        # Segment assignment
        segment = int(kmeans.predict(X_base)[0])
        row_seg = row_e.copy()
        row_seg["taste_segment"] = segment
        X_seg = preprocessor_seg.transform(row_seg)

        # Predictions
        churn_prob = float(xgb_clf.predict_proba(X_seg)[0, 1])
        watch_time = float(xgb_reg.predict(X_base)[0])

        risk_label = (
            "HIGH"   if churn_prob >= 0.5 else
            "MEDIUM" if churn_prob >= 0.2 else
            "LOW"
        )

        return PredictionResponse(
            churn_probability              = round(churn_prob, 4),
            churn_risk_label               = risk_label,
            expected_weekly_watch_time_min = round(max(watch_time, 0), 1),
            taste_segment                  = segment,
            model_version                  = VERSION,
            timestamp                      = datetime.utcnow().isoformat()
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/retrain_trigger", tags=["monitoring"])
def retrain_trigger(psi_threshold: float = 0.25):
    """
    Placeholder retraining trigger.
    In production, this would:
    1. Compute PSI on latest production data
    2. If PSI > threshold, enqueue a retraining job
    3. Return job ID and estimated completion time
    """
    return {
        "message"        : "Retraining job queued (stub — integrate with MLflow Projects)",
        "psi_threshold"  : psi_threshold,
        "timestamp"      : datetime.utcnow().isoformat()
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('streamstay_api.py', 'w') as f:
    f.write(fastapi_code)

print('✅ FastAPI app written → streamstay_api.py')

In [ ]:
# ── W12.2 · Dockerfile ────────────────────────────────────────────────────────
dockerfile = '''FROM python:3.11-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy artefacts and API
COPY models/ ./models/
COPY streamstay_api.py .

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
  CMD curl -f http://localhost:8000/health || exit 1

# Run the API
CMD ["uvicorn", "streamstay_api:app", "--host", "0.0.0.0", "--port", "8000"]
'''

requirements = '''fastapi==0.115.0
uvicorn==0.30.0
pydantic==2.7.0
numpy==1.26.4
pandas==2.2.2
scikit-learn==1.5.0
xgboost==2.0.3
joblib==1.4.2
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)
with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('✅ Dockerfile + requirements.txt written.')
print('\nTo build and run:')
print('  docker build -t streamstay-api .')
print('  docker run -p 8000:8000 streamstay-api')

In [ ]:
# ── W12.3 · Local smoke test (without Docker) ─────────────────────────────────
# Re-load artefact and simulate one prediction
bundle = joblib.load(artefact_path)

test_subscriber = {
    'age': 28, 'country': 'India', 'platform': 'Android',
    'device_type': 'Mobile', 'subscription_type': 'Free',
    'top_genre': 'Pop', 'top_artist': 'Avicii',
    'daily_listening_minutes': 15.0, 'songs_per_day': 3,
    'sessions_per_week': 2, 'avg_song_duration_sec': 180,
    'skip_rate_pct': 38.0, 'playlists_created': 1,
    'likes_per_week': 2, 'unique_artists_per_week': 3,
    'offline_usage_pct': 0.0, 'recommendation_rating': 2.1,
    'days_since_last_active': 30, 'subscription_months': 1,
    'monthly_fee': 0.0, 'auto_renew': 0,
    'payment_failures': 2, 'previous_cancellations': 1
}

row = pd.DataFrame([test_subscriber])
row['engagement_score']  = row['daily_listening_minutes']*0.4 + row['likes_per_week']*5 + row['playlists_created']*10 - row['skip_rate_pct']*2 - row['days_since_last_active']*3
row['loyalty_index']     = row['subscription_months']*row['auto_renew'] - row['payment_failures']*10 - row['previous_cancellations']*15
row['session_density']   = row['songs_per_day'] / (row['sessions_per_week']/7 + 1e-3)
row['catalogue_breadth'] = row['unique_artists_per_week'] * row['playlists_created']
row['is_paid']           = (row['monthly_fee'] > 0).astype(int)

pre_e   = bundle['preprocessor_e']
pre_seg = bundle['preprocessor_seg']
km      = bundle['kmeans']
clf     = bundle['xgb_clf']
reg     = bundle['xgb_reg']

X_base = pre_e.transform(row)
seg_id = int(km.predict(X_base)[0])
row['taste_segment'] = seg_id
X_seg_row = pre_seg.transform(row)

churn_prob = clf.predict_proba(X_seg_row)[0, 1]
watch_time = max(reg.predict(X_base)[0], 0)
risk       = 'HIGH' if churn_prob >= 0.5 else 'MEDIUM' if churn_prob >= 0.2 else 'LOW'

print('🔮 PREDICTION RESULT')
print('─'*40)
print(f'  Churn Probability  : {churn_prob:.4f}')
print(f'  Churn Risk Label   : {risk}')
print(f'  Expected Watch-Time: {watch_time:.1f} min/week')
print(f'  Taste Segment      : {seg_id}')
print(f'  Model Version      : {bundle["version"]}')
print('─'*40)
print('✅ W12 complete — End-to-end prediction pipeline validated.')

---
## 🏁 Final Dashboard — StreamStay Summary

In [ ]:
# ── Final Summary Dashboard ───────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#F8F9FA')
gs  = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

# 1. Model leaderboard (text)
ax1 = fig.add_subplot(gs[0, 0])
ax1.axis('off')
table_data = [
    ['Model',           'AUC'],
    ['Logistic Reg',    f'{auc_lr:.3f}'],
    ['Decision Tree',   f'{auc_dt:.3f}'],
    ['Random Forest',   f'{auc_rf:.3f}'],
    ['XGBoost ★',       f'{test_auc:.3f}'],
]
t = ax1.table(cellText=table_data[1:], colLabels=table_data[0],
              loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(10)
t.scale(1.2, 1.5)
# Highlight champion
for j in range(2):
    t[4, j].set_facecolor('#4C72B0')
    t[4, j].set_text_props(color='white', fontweight='bold')
ax1.set_title('Churn Classifier Leaderboard', fontweight='bold', pad=10)

# 2. Churn distribution
ax2 = fig.add_subplot(gs[0, 1])
churn_rate_by_sub = df.groupby('subscription_type')['churn_next_30_days'].mean() * 100
churn_rate_by_sub.sort_values().plot(kind='barh', ax=ax2,
    color=sns.color_palette('muted', 4), edgecolor='white')
ax2.set_xlabel('Churn %')
ax2.set_title('Churn by Subscription Type', fontweight='bold')
ax2.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))

# 3. Watch-time by segment
ax3 = fig.add_subplot(gs[0, 2])
seg_wt = df.groupby('taste_segment')['expected_weekly_listening_minutes'].mean()
seg_wt.plot(kind='bar', ax=ax3,
    color=sns.color_palette('tab10', BEST_K), edgecolor='white')
ax3.set_xlabel('Segment'); ax3.set_ylabel('Avg min/week')
ax3.set_title('Watch-Time by Taste Segment', fontweight='bold')
ax3.tick_params(axis='x', rotation=0)

# 4. ROC curve comparison
ax4 = fig.add_subplot(gs[1, 0:2])
for name, proba, lw in [
    ('Logistic Reg',  y_prob_lr,  1.5),
    ('Decision Tree', y_prob_dt,  1.5),
    ('Random Forest', y_prob_rf,  1.5),
    ('XGBoost ★',     y_prob_test, 2.5)
]:
    fpr_c, tpr_c, _ = roc_curve(y_clf_test if name == 'XGBoost ★' else y_clf_val,
                                  proba)
    auc_c = roc_auc_score(y_clf_test if name == 'XGBoost ★' else y_clf_val, proba)
    ax4.plot(fpr_c, tpr_c, lw=lw, label=f'{name} AUC={auc_c:.3f}')
ax4.plot([0,1],[0,1],'k--',lw=1, label='Random')
ax4.set_xlabel('FPR'); ax4.set_ylabel('TPR')
ax4.set_title('ROC Curves — All Classifiers', fontweight='bold')
ax4.legend(fontsize=9)

# 5. Metric summary text
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
summary_txt = (
    f'🏆 FINAL TEST METRICS\n\n'
    f'CHURN PREDICTION\n'
    f'  AUC     : {test_auc:.4f}\n'
    f'  F1      : {test_f1:.4f}\n'
    f'  Avg Prec: {test_ap:.4f}\n'
    f'  Brier   : {test_bs:.4f}\n\n'
    f'WATCH-TIME\n'
    f'  RMSE : {test_rmse:.1f} min\n'
    f'  R²   : {test_r2:.4f}\n\n'
    f'Taste Segments : {BEST_K}\n'
    f'Model version  : {VERSION}'
)
ax5.text(0.05, 0.95, summary_txt, transform=ax5.transAxes,
         fontsize=10.5, verticalalignment='top',
         fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#E8F4FD', alpha=0.8))

# 6. Top SHAP features
ax6 = fig.add_subplot(gs[2, 0:2])
shap_mean = np.abs(shap_values.values).mean(axis=0)
shap_series = pd.Series(shap_mean, index=feature_names_all).sort_values(ascending=False).head(12)
shap_series.sort_values().plot(kind='barh', ax=ax6,
    color='#4C72B0', edgecolor='white')
ax6.set_xlabel('Mean |SHAP| value')
ax6.set_title('Top 12 Features — SHAP Global Importance', fontweight='bold')

# 7. Actual vs Predicted watch-time
ax7 = fig.add_subplot(gs[2, 2])
ax7.scatter(y_reg_test, y_reg_pred_test, alpha=0.4, color='#4C72B0', s=18)
mn_t, mx_t = y_reg_test.min(), y_reg_test.max()
ax7.plot([mn_t, mx_t], [mn_t, mx_t], 'r--', lw=1.5)
ax7.set_xlabel('Actual (min/week)'); ax7.set_ylabel('Predicted (min/week)')
ax7.set_title(f'Watch-Time: Actual vs Predicted\nRMSE={test_rmse:.1f}, R²={test_r2:.3f}',
              fontweight='bold')

plt.suptitle('🎵 StreamStay — Final ML Dashboard', fontsize=16,
             fontweight='bold', y=1.01)
plt.savefig('streamstay_final_dashboard.png', bbox_inches='tight', dpi=130)
plt.show()
print('\n🎉 ALL 12 WEEKS COMPLETE! StreamStay ML system built end-to-end.')

---

## 📋 Project Files Summary

| File | Description |
|------|-------------|
| `StreamStay_ChurnAnalytics.ipynb` | This notebook — full 12-week build |
| `streamstay_music_1200.csv` | Source dataset (1200 subscribers) |
| `models/streamstay_champion_v*.pkl` | Versioned model artefact bundle |
| `streamstay_api.py` | FastAPI deployment endpoint |
| `Dockerfile` | Container definition |
| `requirements.txt` | Python dependencies |
| `w1_target_distributions.png` | W1 EDA plots |
| `w2_categorical_eda.png` | W2 categorical breakdowns |
| `w2_correlation_heatmap.png` | W2 correlation matrix |
| `w3_regression_baseline.png` | W3 OLS baseline |
| `w4_lr_curves.png` | W4 ROC + PR curves |
| `w6_decision_tree.png` | W6 tree visualisation |
| `w7_permutation_importance.png` | W7 RF permutation importance |
| `w8_shap_summary.png` | W8 SHAP beeswarm |
| `w9_pca_segments.png` | W9 taste segment PCA |
| `w10_test_evaluation.png` | W10 sealed test results |
| `w11_psi_drift.png` | W11 PSI drift guard |
| `streamstay_final_dashboard.png` | Final summary dashboard |

---

> **StreamStay** — *Predict who's about to cancel — and how much they'll watch.*